# Solver Settings, Saving and Loading

So far, all notebooks called `esM.optimize(timeSeriesAggregation=True)` and let FINE choose everything else. This notebook shows the settings that control the solver, how to check what happened during the optimization, and how to save an optimized model to a file and load it again:

| Topic | Parameters and functions |
|---|---|
| Choosing a solver | `solver`, `fn.ImplementedSolvers.STANDARD_SOLVER` |
| Solver settings | `threads`, `timeLimit`, `optimizationSpecs`, `logFileName` |
| Less output | `verboseLogLevel` |
| What happened? | `esM.solverSpecs`, `esM.objectiveValue` |
| Saving and loading | `fn.xrIO.writeEnergySystemModelToNetCDF`, `fn.xrIO.readNetCDFtoEnergySystemModel` |

In [1]:
import warnings
from pathlib import Path

import fine as fn
from tutorialHelpers import buildBasicTutorialModel

warnings.filterwarnings("ignore")  # keeps the output of this tutorial readable

esM = buildBasicTutorialModel()
esM.aggregateTemporally(n_clusters=8, segments=None)

## 1. Choosing a solver

FINE builds the optimization problem with [Pyomo](https://www.pyomo.org/) and hands it to a solver. Two solvers are installed together with FINE:

- [**HiGHS**](https://highs.dev/): free and open source.
- [**Gurobi**](https://www.gurobi.com/): commercial and usually faster, especially for large problems and problems with integer variables. It needs a license (free licenses are available for academic use). Without a license, Gurobi only solves very small problems.

The parameter `solver` of `esM.optimize` selects one, e.g. `solver="highs"` or `solver="gurobi"`. If it is not given, FINE uses its **standard solver**: Gurobi if it is installed with a full license, HiGHS otherwise. The environment variable `FINE_SOLVER` can set the standard solver explicitly, e.g. on a computer cluster. On this computer, the standard solver is:

In [2]:
standardSolver = fn.ImplementedSolvers.STANDARD_SOLVER.value
standardSolver

'gurobi'

## 2. Solver settings

Four parameters of `esM.optimize` control the solver:

- **`timeLimit`**: the maximum solve time in seconds. Useful on computer clusters, where every job has a time limit. Choose it generously: if the solver stops at the time limit, FINE may not process any results at all, depending on what the solver reports (check `esM.solverSpecs`).
- **`optimizationSpecs`**: further settings, written as one string of `name=value` pairs separated by spaces. The names are those of the solver, so they differ between solvers; see the documentation of [Gurobi](https://docs.gurobi.com/projects/optimizer/en/current/reference/parameters.html) and [HiGHS](https://ergo-code.github.io/HiGHS/dev/options/definitions/).
- **`threads`**: the number of processor threads Gurobi may use (default 3; 0 means all).
- **`logFileName`**: a file into which Gurobi writes its log.

Here, we ask both solvers to use their interior point method (barrier method), which is often the fastest method for large linear problems. The setting has a different name in each solver:

In [3]:
if standardSolver == "gurobi":
    solverSettings = "Method=2"  # 2 = barrier
else:
    solverSettings = "solver=ipm"  # interior point method

## 3. Less output with `verboseLogLevel`

While optimizing, FINE prints what it is doing, and the solver prints its log. How much is printed is set by `verboseLogLevel` of the `EnergySystemModel`:

- `0`: everything,
- `1`: only warnings of FINE and the log of the solver (used in all notebooks),
- `2`: no messages of FINE; for Gurobi, the log of the solver is switched off as well.

It can also be changed later, as an attribute of the model:

In [4]:
esM.verboseLogLevel = 2

esM.optimize(
    timeSeriesAggregation=True,
    solver=standardSolver,
    timeLimit=600,  # seconds
    optimizationSpecs=solverSettings,
)

Set parameter OutputFlag to value 1


Set parameter TimeLimit to value 600


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter Method to value 2


Set parameter LogToConsole to value 0


## 4. What happened during the optimization?

`esM.solverSpecs` collects information about the last optimization: the solver and settings used, the times in seconds needed to build the problem (`buildtime`), to solve it (`solvetime`) and for the whole optimization including the processing of the results (`runtime`), and the result of the solver. `terminationCondition` should be `optimal`; if it is not, e.g. because the time limit was reached or the problem is infeasible, the results have to be treated with care.

In [5]:
esM.solverSpecs

{'solver': 'gurobi',
 'optimizationSpecs': 'Method=2',
 'hasTSA': True,
 'buildtime': 1.2234416007995605,
 'solvetime': 1.990652322769165,
 'runtime': 6.836468458175659,
 'timeLimit': 600,
 'threads': 3,
 'logFileName': '',
 'status': 'ok',
 'terminationCondition': 'optimal'}

The value of the objective function, the total discounted cost of the system in `costUnit`, is stored in `esM.objectiveValue`:

In [6]:
round(esM.objectiveValue)  # million Euro

597390

## 5. Saving and loading a model

Optimizing a large model can take hours, so it is worth saving the result. `fn.xrIO.writeEnergySystemModelToNetCDF` writes the model, i.e. its components with all their inputs and the results of the optimization, into a [NetCDF](https://www.unidata.ucar.edu/software/netcdf/) file. `overwriteExisting=True` replaces an existing file with the same name.

In [7]:
fileName = Path("basic_tutorial_model.nc")
fn.xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=fileName, overwriteExisting=True
)

`fn.xrIO.readNetCDFtoEnergySystemModel` creates a new `EnergySystemModel` from the file. It contains all components and the results, so the optimization summaries are available without optimizing again:

In [8]:
loadedEsM = fn.xrIO.readNetCDFtoEnergySystemModel(fileName)
loadedEsM.getOptimizationSummary("ConversionModel", outputLevel=2, ip=2040)

north         south
Component        Property         Unit                                    
Gas power plants NPVcontribution  [1e6 Euro]     3946.072168   3677.175188
                 TAC              [1e6 Euro/a]   2537.982302   2365.036712
                 capacity         [GW_el]          32.647969     30.423240
                 capexCap         [1e6 Euro/a]   1885.022926   1756.571912
                 operation        [GW_el*h]     28133.593136  31567.899401
                 operation_annual [GW_el*h/a]   28133.593136  31567.899401
                 opexCap          [1e6 Euro/a]    652.959375    608.464800

Not everything is restored, though: `objectiveValue` and `solverSpecs` are not part of the file, and neither is the `balanceLimit` of the model (here the CO<sub>2</sub> limit). A loaded model is therefore meant for looking at results; to optimize again, build the model with the code that created it.

We delete the file again, so that the folder stays clean (skip this cell to keep it):

In [9]:
fileName.unlink()

## Things to try

- Optimize with the other solver (`solver="highs"` or `solver="gurobi"`, if you have a license) and compare `solvetime` in `esM.solverSpecs`.
- Set a very short `timeLimit`, e.g. 1 second. What does `esM.solverSpecs` say?
- Set `verboseLogLevel` back to 1 and optimize again. Which parts of the solver log can you recognize (e.g. the size of the problem and the objective value)?

The next notebook, [Results and Plots](33_results_and_plots.ipynb), goes through the results of an optimization and the plotting functions of FINE.